# Lab 3 — Linear Regression and Stochastic Gradient Descent
## Combined Cycle Power Plant dataset

## How to work in this lab

For the main investigations use the cycle:

**PREDICT → RUN & OBSERVE → EXPLAIN → EXPERIMENT**

**Before** an investigation, write your prediction on the Thinking Sheet **then** run the relevant code. Work in pairs.

Content:

**Investigation 1 and 2**: Intro and recap of concepts from previous labs

**Investigation 3-4**: Linear regression

**Investigation 5**: Feature expansion (Optional. Do it if you have time)

In **Investigation 6** you will implement Stochastic Gradient Descent (SGD) for linear regression yourself. The part you have to write is delimited by a clearly visible banner:

```
# =====================  YOUR CODE STARTS HERE  =====================
...
# ======================  YOUR CODE ENDS HERE  ======================
```

## The task

The dataset contains measurements collected over several years from a **combined cycle power plant** working at full load. Each row is one hourly average observation with four ambient measurements and the quantity we want to predict:

| name | meaning | unit |
|------|---------|------|
| `AT` | ambient temperature | °C |
| `V`  | exhaust vacuum | cm Hg |
| `AP` | ambient pressure | millibar |
| `RH` | relative humidity | % |
| `PE` | **net hourly electrical energy output** (the *target*) | MW |

So this is a <u>regression</u> task: the label $y$ is a **real number**, not a class. We want to learn a function
$$ h_{\mathbf{w}}(\mathbf{x}) = w_0 + w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4 = \langle \mathbf{w}, \mathbf{x}\rangle $$
(where the instance $\mathbf{x}$ has been extended with a constant component equal to $1$), chosen inside the class of linear predictors by **Empirical Risk Minimization with the squared loss**.

Original source: [UCI Machine Learning Repository — Combined Cycle Power Plant](https://archive.ics.uci.edu/ml/datasets/combined+cycle+power+plant). The copy we use has been cleaned.

In [ ]:
# -- numerical computing
import numpy as np
# -- plotting
import matplotlib.pyplot as plt
# -- data handling
import pandas as pd

import sklearn
print('scikit-learn version:', sklearn.__version__)
print('numpy version:       ', np.__version__)

# -- the linear algebra routines we are going to use
from numpy.linalg import solve, lstsq, pinv, cond, norm

Let's load the data. **Note**: the file uses the semicolon `;` as separator, so we have to say so explicitly.

In [ ]:
# -- change the path if the file is not in the same folder as this notebook
DATA_FILE: str = 'ccpp_Data_clean2018.csv'

data: pd.DataFrame = pd.read_csv(DATA_FILE, sep=';')

print('Shape of the dataset (rows, columns):', data.shape)
print('\nColumn names:', list(data.columns))
print('\nFirst rows:')
print(data.head())

Let's look at the data a bit more closely before doing anything else. This is always a good idea: a quick look at the ranges of the features already tells us something important.

In [ ]:
# -- summary statistics of every column
print(data.describe())

# -- are there missing values?
print('\nNumber of missing values:', data.isnull().sum().sum())

# -- are there repeated rows?
print('Number of duplicated rows:', data.duplicated().sum())

Two things are worth noticing already:

1. the four features live on **completely different scales** (`AP` is around $1013$, while `AT` is around $20$);
2. the target `PE` varies by about $75$ MW between its minimum and its maximum.

Let's also look at how each feature is related to the target. The **(Pearson) correlation coefficient** between a feature and the target is a number in $[-1,1]$; it measures how well the relation between the two is described by a *straight line*. It is only a first, rough indication (it says nothing about non-linear relations, and it looks at one feature at a time), but it is cheap to compute.

In [ ]:
# -- correlation of each column with every other column
print(data.corr().round(3))

print('\nCorrelation of each feature with the target PE:')
print(data.corr()['PE'].round(3))

In [ ]:
# -- a visual version of the same thing: each feature against the target
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, feature in zip(axes, ['AT', 'V', 'AP', 'RH']):
    ax.scatter(data[feature], data['PE'], s=2, alpha=0.3)
    ax.set_xlabel(feature)
    ax.set_ylabel('PE')
plt.tight_layout()
plt.show()

For convenience, let's store the instances (vectors of features) in the matrix $\mathbf{X}$ and the targets in the vector $\mathbf{y}$.

In [ ]:
feature_names: list = ['AT', 'V', 'AP', 'RH']

X: np.ndarray = data[feature_names].to_numpy(dtype=float)
y: np.ndarray = data['PE'].to_numpy(dtype=float)

print('X has shape', X.shape, '-> m =', X.shape[0], 'samples and d =', X.shape[1], 'features')
print('y has shape', y.shape)
print('\nFirst 3 instances:\n', X[:3])
print('\nFirst 3 targets:  ', y[:3])

# Investigation 1 — Splitting into training set and test set

### PREDICT — do not run the next cells yet

1. We are going to keep $70\%$ of the data for training and $30\%$ for testing. Do you expect the error measured on the **training** set to be smaller or larger than the error measured on the **test** set? Must this ordering hold in *every* experiment?

Write your answers on the Thinking Sheet before running the code.

As in Lab 2, we first permute the rows at random and only then cut the data in two. Since we use randomization, we fix a random seed so that the whole procedure is **reproducible**.

Use **your own ID number** as the seed: your numbers will then differ slightly from those of your neighbours, which is exactly the point — the *conclusions* must be the same, the *digits* need not be.

In [ ]:
# -- set the random seed to your ID number
IDnumber: int = 1
np.random.seed(IDnumber)

m: int = X.shape[0]

# -- a random ordering of the indices 0, 1, ..., m-1
permutation = np.random.permutation(m)

# -- apply the SAME permutation to the rows of X and to the entries of y,
#    so that instance i keeps its own target
X_perm: np.ndarray = X[permutation]
y_perm: np.ndarray = y[permutation]

print('First 10 permuted indices:', permutation[:10])

In [ ]:
# -- train/test split: 70% training, 30% test
fraction_train: float = 0.7

m_train: int = int(m * fraction_train)
m_test: int = m - m_train
print('Number of samples in the training set:', m_train)
print('Number of samples in the test set:    ', m_test)

X_train: np.ndarray = X_perm[:m_train, :]
y_train: np.ndarray = y_perm[:m_train]

X_test: np.ndarray = X_perm[m_train:, :]
y_test: np.ndarray = y_perm[m_train:]

print('\nX_train:', X_train.shape, ' X_test:', X_test.shape)

### RUN & OBSERVE → EXPLAIN → EXPERIMENT

Compare the two halves of the original file. In **this** dataset the rows do not appear to be sorted by target value, so an unshuffled split would not have been catastrophic.

**Explain.** Why do we shuffle anyway? What would you have to check, *every single time*, in order to be allowed to skip the shuffling? Which of the two costs more: shuffling, or checking?

**Experiment.** Build a second split using the first $70\%$ of the rows of the *unpermuted* `X` and `y`. Keep it aside: at the end of Investigation 4 you will be able to compare the test $R^2$ obtained with the two splits. Predict first whether they will be close.

# Investigation 2 — Scaling, using the training data only

### PREDICT — do not run the checks yet

We will compute the mean and the standard deviation of each feature **on the training set only**, and use them to standardize both the training set and the test set:
$$ z_j = \frac{x_j - \widehat{\mu}_j}{\widehat{\sigma}_j}, \qquad \widehat{\mu}_j, \widehat{\sigma}_j \ \text{computed on the training set.} $$

1. After this transformation, what will be the mean and the standard deviation of each feature **in the scaled training set**? Give exact numbers.
2. And in the scaled **test** set? Exactly the same numbers? Approximately? Why?
3. The features have very different ranges (`AP` $\approx 1013$, `AT` $\approx 20$). Does that make the system of normal equations harder to solve numerically? (Keep the word *condition number* in mind.)

In [ ]:
from sklearn.preprocessing import StandardScaler

# -- we "learn" the scaling function using the TRAINING data only
scaler = StandardScaler().fit(X_train)

# -- we then apply that same function to both training and test data:
#    this simulates what happens when future data arrive one at a time
X_train_scaled: np.ndarray = scaler.transform(X_train)
X_test_scaled: np.ndarray = scaler.transform(X_test)

print('Means learned from the training set: ', np.round(scaler.mean_, 4))
print('Stds  learned from the training set: ', np.round(scaler.scale_, 4))

As in Lab 2, let's do exactly the same thing "by hand", to see that there is no magic in `StandardScaler`.

In [ ]:
X_means: np.ndarray = np.mean(X_train, axis=0)
X_stds: np.ndarray = np.std(X_train, axis=0)

X_train_scaled_byhand: np.ndarray = (X_train - X_means) / X_stds
X_test_scaled_byhand: np.ndarray = (X_test - X_means) / X_stds

print('Same result as the scaler on the training set?', np.allclose(X_train_scaled, X_train_scaled_byhand))
print('Same result as the scaler on the test set?    ', np.allclose(X_test_scaled, X_test_scaled_byhand))

In [ ]:
print('Scaled TRAINING set — mean of each feature:', np.round(X_train_scaled.mean(axis=0), 6))
print('Scaled TRAINING set — std  of each feature:', np.round(X_train_scaled.std(axis=0), 6))
print()
print('Scaled TEST set     — mean of each feature:', np.round(X_test_scaled.mean(axis=0), 4))
print('Scaled TEST set     — std  of each feature:', np.round(X_test_scaled.std(axis=0), 4))

### RUN & OBSERVE → EXPLAIN → EXPERIMENT

The scaled **training** features have mean exactly $0$ and standard deviation exactly $1$ (up to floating-point error). The scaled **test** features do not: their means are small but non-zero and their standard deviations are close to, but not exactly, $1$.

**Explain.** Why is that not a bug but the *correct* behaviour? What would it mean if the test features came out with mean exactly $0$?

The scaling function is **part of the model**: it was estimated from data, it has parameters ($\widehat{\mu}_j, \widehat{\sigma}_j$), and at prediction time it must be applied to a new instance exactly as it was applied during training.

**Experiment.** Fit a `StandardScaler` on the *whole* dataset `X` and use it to transform training and test data. The code runs perfectly. Explain why the procedure is nevertheless methodologically wrong, and what kind of over-optimistic estimate it produces.

# Investigation 3 — The closed-form solution, and what scaling does to it

Recall from class. We collect the training instances, each extended with a leading component equal to $1$, in the **design matrix** $\mathbf{X} \in \mathbb{R}^{m \times (d+1)}$, and the targets in $\mathbf{y} \in \mathbb{R}^{m}$. ERM with the squared loss means minimizing
$$ RSS(\mathbf{w}) \;=\; \sum_{i=1}^{m} \big(\langle \mathbf{w}, \mathbf{x}_i\rangle - y_i\big)^2 \;=\; \lVert \mathbf{X}\mathbf{w} - \mathbf{y}\rVert^2 .$$

Setting the gradient $\nabla RSS(\mathbf{w}) = 2\,\mathbf{X}^T(\mathbf{X}\mathbf{w} - \mathbf{y})$ to zero gives the **normal equations**
$$ \mathbf{X}^T\mathbf{X}\,\mathbf{w} \;=\; \mathbf{X}^T\mathbf{y}. $$
When $\mathbf{X}^T\mathbf{X}$ is invertible the solution is unique, $\mathbf{w}^\star = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$; in general $\mathbf{w}^\star = \mathbf{X}^{+}\mathbf{y}$, with $\mathbf{X}^{+}$ the pseudoinverse obtained from the SVD.

### PREDICT — do not run the next cells yet

1. What is the **condition number** of the design matrix built from the raw features, roughly? And from the scaled features? Order of magnitude is enough.
2. We will solve the normal equations twice: once with the raw features, once with the scaled features. Will the two **coefficient vectors** be the same?
3. Will the two sets of **predictions on the test set** be the same? Careful — this is a different question, and in Lab 2 the answer for the Perceptron was *no*.

### EXPERIMENT 
First we build the design matrix, i.e. we add the column of ones that accounts for the intercept $w_0$. 

Then we compute the least square solution

In [ ]:
# -- returns A with a leading column of ones, so that w[0] plays the role of the intercept
def add_intercept_column(A: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=float)
    return np.hstack([np.ones((A.shape[0], 1)), A])

# -- design matrices: raw features and scaled features
D_train_raw = add_intercept_column(X_train)
D_test_raw = add_intercept_column(X_test)

D_train_scaled = add_intercept_column(X_train_scaled)
D_test_scaled = add_intercept_column(X_test_scaled)

print('Design matrix shape:', D_train_raw.shape, ' (m_train rows, d+1 columns)')
print('\nFirst row, raw:   ', np.round(D_train_raw[0], 3))
print('First row, scaled:', np.round(D_train_scaled[0], 3))

In [ ]:
print('cond(X)  with RAW    features:', f'{cond(D_train_raw):.4g}')
print('cond(X)  with SCALED features:', f'{cond(D_train_scaled):.4g}')
print()
print('cond(X^T X) with RAW    features:', f'{cond(D_train_raw.T @ D_train_raw):.4g}')
print('cond(X)^2   with RAW    features:', f'{cond(D_train_raw) ** 2:.4g}')
print()
print('cond(X^T X) with SCALED features:', f'{cond(D_train_scaled.T @ D_train_scaled):.4g}')
print('cond(X)^2   with SCALED features:', f'{cond(D_train_scaled) ** 2:.4g}')

### Three ways to compute the least-squares solution

Our original problem is

$$
\min_{\mathbf{w}}
\|\mathbf{y}-\mathbf{X}\mathbf{w}\|_2^2.
$$

We now compare three computational procedures.

#### Method 1 — Form and solve the normal equations

In class, we derived the optimality condition

$$
\mathbf{X}^T\mathbf{X}\mathbf{w}
=\mathbf{X}^T\mathbf{y}.
$$

Defining

$$
\mathbf{A}=\mathbf{X}^T\mathbf{X},
\qquad
\mathbf{b}=\mathbf{X}^T\mathbf{y},
$$

we obtain the square linear system

$$
\mathbf{A}\mathbf{w}=\mathbf{b}.
$$

The Python command

`np.linalg.solve(X.T @ X, X.T @ y)`

performs two main operations:

1. It explicitly constructs the matrix $\mathbf{A}$ and
   the vector $\mathbf{b}$.
2. It solves the square linear system
   $\mathbf{A}\mathbf{w}=\mathbf{b}$.

The solver does not explicitly compute $\mathbf{A}^{-1}$.
Instead, it uses a numerical matrix factorization to solve
the system.

**When does this work?**

If $\mathbf{X}$ has full column rank, then $\mathbf{A}$
is invertible and the normal equations have a unique solution.

If $\mathbf{X}$ is rank-deficient, $\mathbf{A}$ is singular,
and `solve` is not an appropriate method.

**Numerical issue:** constructing $\mathbf{X}^T\mathbf{X}$
squares the condition number and can reduce numerical accuracy.

#### Method 2 — Solve the least-squares problem directly

The Python command

`np.linalg.lstsq(X, y, rcond=None)[0]`

starts directly from the original minimization problem:

$$
\min_{\mathbf{w}}
\|\mathbf{y}-\mathbf{X}\mathbf{w}\|_2^2.
$$

Unlike Method 1, it does not explicitly construct
$\mathbf{X}^T\mathbf{X}$.

Instead, a numerical least-squares solver uses an appropriate
matrix factorization to compute the solution.
For example an SVD-based solver using the singular values
and singular vectors of $\mathbf{X}$.


**Important:** NumPy's `lstsq` returns a minimum-norm
least-squares solution, including when the problem is
rank-deficient, subject to its numerical rank tolerance.

The key advantage is that we work directly with
$\mathbf{X}$ rather than explicitly forming
$\mathbf{X}^T\mathbf{X}$.

#### Method 3 — Apply the pseudoinverse

In class, we derived the minimum-norm least-squares solution:

$$
\mathbf{w}^\star=\mathbf{X}^+\mathbf{y}.
$$

The Python command

`np.linalg.pinv(X) @ y`

first computes the pseudoinverse $\mathbf{X}^+$,
using the SVD, and then multiplies it by $\mathbf{y}$.

Recall that we also proved

$$
\mathbf{X}^+\mathbf{y}
=
\mathbf{A}^+\mathbf{X}^T\mathbf{y},
\qquad
\mathbf{A}=\mathbf{X}^T\mathbf{X}.
$$

**Note:** Both methods 2 and 3 use the SVD, but `lstsq` applies its factors directly to $\mathbf{y}$ without explicitly constructing $\mathbf{X}^+$, whereas `pinv(X) @ y` first constructs the pseudoinverse and then multiplies it by $\mathbf{y}$.

### Before running the code

1. Which procedure is most likely to suffer from numerical errors when $\mathbf{X}$ is ill-conditioned? Why?
3. Why might Methods 2 and 3 produce slightly different numerical results, despite representing the same mathematical solution?


In [ ]:
def least_squares_three_ways(A: np.ndarray, t: np.ndarray) -> None:
    w_normal = solve(A.T @ A, A.T @ t)      # normal equations
    w_lstsq = lstsq(A, t, rcond=None)[0]    # least squares solver
    w_pinv = pinv(A) @ t                    # pseudoinverse from the SVD

    print('  normal equations :', np.round(w_normal, 6))
    print('  lstsq            :', np.round(w_lstsq, 6))
    print('  pseudoinverse    :', np.round(w_pinv, 6))
    print('  max |normal - lstsq| =', f'{np.abs(w_normal - w_lstsq).max():.3e}')
    print('  max |pinv   - lstsq| =', f'{np.abs(w_pinv - w_lstsq).max():.3e}')
    return w_lstsq

print('RAW features:')
w_raw = least_squares_three_ways(D_train_raw, y_train)
print('\nSCALED features:')
w_scaled = least_squares_three_ways(D_train_scaled, y_train)

In [ ]:
# -- the two coefficient vectors
print('w with RAW    features:', np.round(w_raw, 6))
print('w with SCALED features:', np.round(w_scaled, 6))

# -- but what about the PREDICTIONS on the test set?
pred_raw = D_test_raw @ w_raw
pred_scaled = D_test_scaled @ w_scaled

print('\nFirst 5 predictions, raw features:   ', np.round(pred_raw[:5], 6))
print('First 5 predictions, scaled features:', np.round(pred_scaled[:5], 6))
print('\nMaximum absolute difference over the whole test set:',
      f'{np.abs(pred_raw - pred_scaled).max():.3e}')

### QUESTION: Why does the condition number matter for numerical accuracy?

### ANSWER

In class, we defined the condition number using the singular values:

$$
\kappa(\mathbf{X})=\frac{\sigma_1}{\sigma_k}.
$$

We also showed, using the SVD, that the eigenvalues of
$\mathbf{A}=\mathbf{X}^T\mathbf{X}$ are the squared singular values
of $\mathbf{X}$ (together with possible zero eigenvalues).

If $\mathbf{X}$ has full column rank,

$$
\kappa(\mathbf{A})
=\frac{\sigma_1^2}{\sigma_k^2}
=\kappa(\mathbf{X})^2.
$$

But why is this important when we solve the problem on a computer?

#### 1. Computers do not perform exact arithmetic

Computers represent real numbers using a finite number of bits.
As a result, operations such as multiplication and addition can
introduce small rounding errors.

Even if the original data are exact, the matrices and intermediate
results computed by an algorithm may contain small numerical errors.

For example, when we compute

`A = X.T @ X`

the entries of $\mathbf{A}$ are obtained through many
multiplications and additions. These operations are performed
in finite precision.

#### 2. The condition number measures sensitivity

Consider a linear system

$$
\mathbf{A}\mathbf{w}=\mathbf{b}.
$$

Suppose the right-hand side is slightly perturbed:

$$
\mathbf{b}\longrightarrow\mathbf{b}+\Delta\mathbf{b}.
$$

The solution changes from $\mathbf{w}$ to
$\mathbf{w}+\Delta\mathbf{w}$.

For an invertible matrix $\mathbf{A}$, we have the bound

$$
\frac{\|\Delta\mathbf{w}\|_2}{\|\mathbf{w}\|_2}
\le
\kappa(\mathbf{A})
\frac{\|\Delta\mathbf{b}\|_2}{\|\mathbf{b}\|_2},
$$

assuming $\mathbf{b}\ne0$.

This tells us that the relative error in the solution can be
much larger than the relative perturbation in the data.

A large condition number means that some small perturbations
can produce much larger changes in the solution.

**Important:** this is a worst-case sensitivity bound.
It does not mean that every perturbation will be amplified
by exactly the condition number.

#### 3. Why do we talk about losing decimal digits?

Suppose a numerical perturbation has relative size approximately

$$
10^{-16}.
$$

This is the order of machine precision in standard
64-bit floating-point arithmetic.

If the condition number is of order $10^5$, a worst-case
amplification by this factor gives

$$
10^5\cdot10^{-16}=10^{-11}.
$$

If the condition number is instead of order $10^{10}$,

$$
10^{10}\cdot10^{-16}=10^{-6}.
$$

The first estimate corresponds to roughly 11 reliable decimal
digits, while the second corresponds to roughly 6.


#### 4. Why is forming the normal equations potentially problematic?

Recall that

$$
\mathbf{A}=\mathbf{X}^T\mathbf{X}.
$$

If $\kappa(\mathbf{X})\approx 2\cdot10^5$, then

$$
\kappa(\mathbf{A})
=\kappa(\mathbf{X})^2
\approx4\cdot10^{10}.
$$

The system involving $\mathbf{A}$ is therefore much more
sensitive to perturbations than a problem whose conditioning
is characterized directly by $\kappa(\mathbf{X})$.

Furthermore, computing $\mathbf{X}^T \mathbf{X}$ itself
introduces rounding errors.

This is why numerical least-squares algorithms generally
avoid explicitly forming the normal equations when
numerical accuracy is important.

### LET'S NOW COMPARE THE THREE SOLVERS

Look first at the three numerical solutions.

On the **scaled data**, the coefficients obtained by the three procedures agree to approximately $10^{-13}$ in the reported comparison.

On the **raw data**, the normal-equations solution shows a larger discrepancy from the solution obtained by `lstsq`.

Recall that, when $\mathbf{X}$ has full column rank,

$$
\kappa(\mathbf{X}^T\mathbf{X})=\kappa(\mathbf{X})^2.
$$

For the raw design matrix, $\kappa(\mathbf{X}^T\mathbf{X})$ is approximately $5\cdot10^{10}$.

Forming the normal equations can therefore amplify the effects of floating-point rounding errors.

The observed numerical discrepancy is consistent with this loss of conditioning.

The direct least-squares solver and the SVD-based pseudoinverse avoid explicitly forming $\mathbf{X}^T\mathbf{X}$.

**Different coefficients, similar predictions**

Now compare the models trained on raw and standardized features.

Their coefficients are different, but their predictions agree to approximately $10^{-13}$ in this experiment.

Why?

Standardization transforms the input features:

$$
\mathbf{z}=\mathbf{S}^{-1}(\mathbf{x}-\boldsymbol{\mu}),
$$

where $\mathbf{S}$ is a diagonal matrix containing the feature standard deviations.

Assume that all standard deviations are nonzero.

A linear predictor trained on standardized features has the form

$$
\hat y=b_z+\mathbf{w}_z^T\mathbf{z}.
$$

Substituting the definition of $\mathbf{z}$ gives

$$
\hat y
=b_z+\mathbf{w}_z^T\mathbf{S}^{-1}
(\mathbf{x}-\boldsymbol{\mu}).
$$

Rearranging,

$$
\hat y
=
\left(b_z-\mathbf{w}_z^T\mathbf{S}^{-1}
\boldsymbol{\mu}\right)
+
\left(\mathbf{S}^{-1}\mathbf{w}_z\right)^T
\mathbf{x}.
$$

Therefore, the same predictor can be expressed in the original coordinates using

$$
\boxed{
\mathbf{w}_x=\mathbf{S}^{-1}\mathbf{w}_z
}
$$

and

$$
\boxed{
b_x=b_z-\mathbf{w}_z^T\mathbf{S}^{-1}
\boldsymbol{\mu}.
}
$$

Thus, standardization changes the numerical values of the coefficients, but not the set of affine functions that the model can represent.

For least squares, if the fitted prediction function is uniquely determined and the optimization is solved accurately, the raw and standardized formulations give the same prediction function.

In our experiment, the predictions agree up to numerical precision.


**Finally a sanity check** against the scikit-learn implementation of Linear Regression Solver. Note that `LinearRegression` adds the intercept itself, so we pass it the features *without* the column of ones.

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X_train_scaled, y_train)

print('scikit-learn intercept:', round(linreg.intercept_, 6))
print('scikit-learn coef_    :', np.round(linreg.coef_, 6))
print('our w (scaled)        :', np.round(w_scaled, 6))
print('\nSame model?', np.allclose(np.r_[linreg.intercept_, linreg.coef_], w_scaled))

# Investigation 4 — How good is the model? $RSS$, $RMSE$ and $R^2$

Three numbers, all computed from the same residuals $r_i = h(\mathbf{x}_i) - y_i$:

* $RSS = \sum_i r_i^2$ — depends on the number of points, so it cannot be compared across sets of different size;
* $RMSE = \sqrt{\frac{1}{m}\sum_i r_i^2}$ — the average error, **in MW**, so it is directly interpretable;
* $R^2 = 1 - \dfrac{RSS}{TSS}$, with $TSS = \sum_i (y_i - \bar{y})^2$ — dimensionless.

Recall the exercise we did in class: $TSS$ is the $RSS$ of the **best constant predictor** $h(\mathbf{x}) = \bar{y}$. So $R^2$ compares our model against the most naive baseline there is, and $R^2 = 0$ means "no better than predicting the average".

### PREDICT — do not run the next cells yet

1. The target `PE` has a standard deviation of about $17$ Megawatt. What $RMSE$ would you expect from a model that is genuinely useful? What $RMSE$ does the constant predictor $\bar{y}$ have?
2. Can $R^2$ measured on the **test** set be negative? Can $R^2$ measured on the **training** set be negative, for the model we just fitted?
3. From the correlations computed at the beginning, guess the $R^2$ of a model that uses `AT` alone.

In [ ]:
def rss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sum((y_pred - y_true) ** 2))

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def r_squared(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    tss = float(np.sum((y_true - np.mean(y_true)) ** 2))
    return 1.0 - rss(y_true, y_pred) / tss

pred_train = D_train_scaled @ w_scaled
pred_test = D_test_scaled @ w_scaled

print('TRAINING    RSS = %12.2f   RMSE = %6.3f MW   R^2 = %.4f'
      % (rss(y_train, pred_train), rmse(y_train, pred_train), r_squared(y_train, pred_train)))
print('TEST        RSS = %12.2f   RMSE = %6.3f MW   R^2 = %.4f'
      % (rss(y_test, pred_test), rmse(y_test, pred_test), r_squared(y_test, pred_test)))

In [ ]:
# -- cross-check with scikit-learn
from sklearn.metrics import r2_score, mean_squared_error

print('Our R^2 on test        :', round(r_squared(y_test, pred_test), 8))
print('sklearn r2_score on test:', round(r2_score(y_test, pred_test), 8))
print('\nOur RMSE on test           :', round(rmse(y_test, pred_test), 8))
print('sklearn sqrt(MSE) on test  :', round(np.sqrt(mean_squared_error(y_test, pred_test)), 8))

Now the baseline. **Careful**: the constant we are allowed to use is the mean of the **training** targets — the test targets are not available when the model is built.

In [ ]:
baseline_train = np.full_like(y_train, y_train.mean())
baseline_test = np.full_like(y_test, y_train.mean())

print('Mean of the training targets: %.4f MW' % y_train.mean())
print()
print('Baseline (predict ybar)  TRAINING  RMSE = %6.3f   R^2 = %.5f'
      % (rmse(y_train, baseline_train), r_squared(y_train, baseline_train)))
print('Baseline (predict ybar)  TEST      RMSE = %6.3f   R^2 = %.5f'
      % (rmse(y_test, baseline_test), r_squared(y_test, baseline_test)))

On the **training** set the baseline has exactly $R^2 = 0$: by definition, $\bar{y}_{\text{train}}$ is the constant that minimizes $RSS$ on the training set, so $RSS = TSS$ there.

On the **test** set the baseline usually has a *slightly negative* $R^2$. There is no contradiction: on the test set, $TSS$ is computed around the **test** mean, while our constant is the **training** mean. A constant that is not the mean of the set on which it is evaluated does worse than that set's own mean. Note how small the effect is — and note that it would not be small if the split were not random.

In [ ]:
# -- how much does each single feature explain, on its own?
print('%-6s %10s %10s' % ('model', 'train R^2', 'test R^2'))
for j, name in enumerate(feature_names):
    A_tr = add_intercept_column(X_train_scaled[:, [j]])
    A_te = add_intercept_column(X_test_scaled[:, [j]])
    w_j = lstsq(A_tr, y_train, rcond=None)[0]
    print('%-6s %10.4f %10.4f' % (name, r_squared(y_train, A_tr @ w_j), r_squared(y_test, A_te @ w_j)))

print('%-6s %10.4f %10.4f' % ('all 4', r_squared(y_train, pred_train), r_squared(y_test, pred_test)))

In [ ]:
# -- how do the residuals look? (a good model leaves no visible structure behind)
residuals_test = pred_test - y_test

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(pred_test, residuals_test, s=3, alpha=0.3)
axes[0].axhline(0.0, color='k', linewidth=0.8)
axes[0].set_xlabel('predicted PE (MW)')
axes[0].set_ylabel('residual = predicted - true')
axes[0].set_title('Residuals vs predictions (test set)')

axes[1].hist(residuals_test, bins=50)
axes[1].set_xlabel('residual (MW)')
axes[1].set_title('Distribution of the residuals (test set)')
plt.tight_layout()
plt.show()

### RUN & OBSERVE → EXPLAIN → EXPERIMENT

**Explain.** The $RMSE$ is around $4.5$–$4.7$ Megawatt against a target standard deviation of about $17$ Megawatt, i.e. $R^2 \approx 0.93$: the linear model removes roughly $93\%$ of the variance of the target. `AT` alone already accounts for about $0.90$. Does that mean the other three features are useless? Look at how much they add, and remember that correlated features can carry overlapping information.

Look at the residual plot. Is the cloud flat, or can you see a curve? A visible curve is evidence that the *linear* class is too small — which is exactly what Investigation 5 (next) is about.

**Experiment.** Remove one feature at a time (fit on the remaining three) and record the test $R^2$. Which feature can be dropped at almost no cost? Predict before running. Then compare the test $R^2$ of the shuffled split with the one you computed in the experiment of Investigation 1.

# Investigation 5 — Feature expansion *(optional — do it if you have time)*

The class of linear predictors is small. We can make it larger **without changing the algorithm at all**, by replacing each instance $\mathbf{x} \in \mathbb{R}^d$ with a richer vector of features
$$ \psi : \mathbb{R}^{d} \to \mathbb{R}^{D}, \qquad \mathbf{x} \mapsto \psi(\mathbf{x}), $$
and then fitting a linear model in $\psi(\mathbf{x})$. The predictor is **non-linear in $\mathbf{x}$ but still linear in the parameters $\mathbf{w}$**, so the normal equations apply verbatim to the new design matrix $\mathbf{\Psi}$.

With all monomials of degree at most $r$ in $d$ variables, $D = \binom{d+r}{r}$. For $d = 4$: $D = 5, 15, 35, 70, 126, \dots$ for $r = 1,2,3,4,5$ (including the constant term).

### PREDICT — do not run the next cells yet

1. As $r$ grows, what will happen to the **training** $R^2$? Must it be monotone? Why?
2. And to the **test** $R^2$?
3. What will happen to $\kappa(\mathbf{\Psi})$? Does it matter whether the monomials are built from the *raw* or from the *scaled* features?

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

print('%3s %6s %14s %12s %12s %12s %12s'
      % ('r', 'D', 'cond(Psi)', 'train R^2', 'test R^2', 'train RMSE', 'test RMSE'))
for r in range(1, 7):
    poly = PolynomialFeatures(degree=r, include_bias=False)
    P_train = add_intercept_column(poly.fit_transform(X_train_scaled))
    P_test = add_intercept_column(poly.transform(X_test_scaled))

    w_poly = lstsq(P_train, y_train, rcond=None)[0]
    p_tr, p_te = P_train @ w_poly, P_test @ w_poly

    print('%3d %6d %14.4g %12.4f %12.4f %12.3f %12.3f'
          % (r, P_train.shape[1], cond(P_train),
             r_squared(y_train, p_tr), r_squared(y_test, p_te),
             rmse(y_train, p_tr), rmse(y_test, p_te)))

Now the same expansion, but built from the **raw** features instead of the scaled ones. We only look at the conditioning.

In [ ]:
print('%3s %6s %14s' % ('r', 'D', 'cond(Psi)'))
for r in range(1, 5):
    poly = PolynomialFeatures(degree=r, include_bias=False)
    P_train_raw = add_intercept_column(poly.fit_transform(X_train))
    print('%3d %6d %14.4g' % (r, P_train_raw.shape[1], cond(P_train_raw)))

This is the ill-conditioning we discussed in class. Raising a feature of size $\sim 10^3$ to the third power produces a column of size $\sim 10^9$ next to a column of ones: by $r = 3$ the condition number is around $10^{15}$. Scaling first is not cosmetic here — it is necessary for computational stability.

Finally, another instructive experiment: the same expansions on a **small** training set.

In [ ]:
# -- take only 60 training points
rng_small = np.random.default_rng(7)
subset = rng_small.choice(m_train, size=60, replace=False)
X_small, y_small = X_train_scaled[subset], y_train[subset]

print('Training on only %d points.\n' % len(subset))
print('%3s %6s %12s %14s' % ('r', 'D', 'train R^2', 'test R^2'))
for r in range(1, 6):
    poly = PolynomialFeatures(degree=r, include_bias=False)
    P_small = add_intercept_column(poly.fit_transform(X_small))
    P_test = add_intercept_column(poly.transform(X_test_scaled))
    w_poly = lstsq(P_small, y_small, rcond=None)[0]
    print('%3d %6d %12.4f %14.4f'
          % (r, P_small.shape[1], r_squared(y_small, P_small @ w_poly),
             r_squared(y_test, P_test @ w_poly)))

### RUN & OBSERVE → EXPLAIN → EXPERIMENT

On the full training set the training $R^2$ **must** increase with $r$ (each class contains the previous one, so the minimum of $RSS$ cannot increase), while the test $R^2$ improves for a while and then starts to degrade: the usual picture.

On the small training set the effect is magnified. As soon as $D + 1$ exceeds the number of training points, the linear system is **underdetermined**: there are infinitely many $\mathbf{w}$ with $RSS = 0$, `lstsq` returns the minimum-norm one (the pseudoinverse solution $\mathbf{\Psi}^{+}\mathbf{y}$ from the SVD slides), the training $R^2$ is exactly $1$ — and the test $R^2$ becomes hugely negative. The model *interpolates* the training data and predicts nonsense everywhere else.

**Explain.** Relate what you see to the size of the hypothesis class and to the amount of training data. Which of the two did we change in this last cell?

**Experiment.** Increase the size of the small training set ($60 \to 120 \to 300 \to 1000$) and find, for each size, the degree $r$ at which the test $R^2$ collapses. What is the pattern?


### RUN & OBSERVE → EXPLAIN → EXPERIMENT

#### 1. What happens when we increase the polynomial degree?

Recall that polynomial regression is still a linear model in its parameters.

We replace the original features with polynomial features, but the predictor still has the form

$$
\hat{\mathbf{y}}=\mathbf{\Psi}\mathbf{w},
$$

where $\mathbf{\Psi}$ is the polynomial design matrix.

As the polynomial degree $r$ increases, the number of columns $D$ of $\mathbf{\Psi}$ increases.

A higher-degree polynomial model can represent all the functions of lower-degree models, plus additional ones.

Therefore, the hypothesis classes are nested:

$$
\mathcal{H}_1\subseteq\mathcal{H}_2
\subseteq\mathcal{H}_3\subseteq\cdots
$$

**What does this imply for training error?**

Since the higher-degree model can always reproduce the solution of a lower-degree model, its minimum training RSS cannot be larger.

Thus, in exact arithmetic,

$$
RSS_{\mathrm{train}}(r+1)
\leq RSS_{\mathrm{train}}(r).
$$

Since

$$
R^2=1-\frac{RSS}{TSS},
$$

and the training set is unchanged, the training $R^2$ cannot decrease as the polynomial degree increases.

However, the test $R^2$ does not have to improve.

A more flexible model may fit the training data better while generalizing worse to new observations.

**Observe:** Does the test $R^2$ initially improve and then deteriorate? At which degree does this happen?

#### 2. What changes when we use only 60 training examples?

In the last experiment, we reduced the number of training examples to

$$
m_{\mathrm{train}}=60.
$$

We are still considering polynomial models, but now we have much less data available to estimate their parameters.

Recall that the polynomial design matrix has dimensions

$$
\mathbf{\Psi}\in\mathbb{R}^{m_{\mathrm{train}}\times D},
$$

where $D$ is the total number of polynomial coefficients, including the intercept.

As the degree increases, $D$ increases rapidly.

**What happens if $D>m_{\mathrm{train}}$?**

There are more unknown parameters than equations.

The linear system

$$
\mathbf{\Psi}\mathbf{w}=\mathbf{y}
$$

is called **underdetermined**.

If $\mathbf{\Psi}$ has full row rank, then every training target vector $\mathbf{y}$ can be represented exactly.

Consequently, there are infinitely many parameter vectors satisfying

$$
\mathbf{\Psi}\mathbf{w}=\mathbf{y}.
$$

All these solutions have

$$
RSS_{\mathrm{train}}=0.
$$

Therefore, provided $TSS_{\mathrm{train}}>0$,

$$
R^2_{\mathrm{train}}=1.
$$

This is called **interpolation**: the model reproduces all the observed training targets exactly.

**Important:** having more parameters than observations does not, by itself, guarantee exact interpolation. We also need the appropriate rank condition.

#### 3. Which solution does `lstsq` return?

Recall the SVD lectures.

When the least-squares problem has multiple solutions, the pseudoinverse selects the solution with minimum Euclidean norm:

$$
\boxed{
\mathbf{w}^{\star}=\mathbf{\Psi}^{+}\mathbf{y}.
}
$$

This is the mathematical solution computed by `lstsq`, up to numerical precision and the numerical rank tolerance.

But selecting the minimum-norm solution does not guarantee good predictions on unseen data.

A model may reproduce all training targets exactly and still perform very poorly on the test set.

In particular, if

$$
RSS_{\mathrm{test}}>TSS_{\mathrm{test}},
$$

then

$$
R^2_{\mathrm{test}}<0.
$$

A negative test $R^2$ means than the constant predictor based on the test-erefore4 poor test performance may reflect statistical overfitting, numerical difficulties, or both.

These are different phenomena and should not be confused.

#### 5. EXPLAIN — Your turn

Use the results printed by the previous cells.

1. Why can the training $R^2$ not decrease when we increase the polynomial degree, assuming exact least-squares solutions?

2. Why can the test $R^2$ decrease even when the training $R^2$ increases?

3. What does it mean for the polynomial system to be underdetermined?

4. Why does `lstsq` select a minimum-norm solution? Does minimum norm guarantee good generalization?

5. In the experiment with 60 training examples, did we cha5ge the hypothesis class, the amount of training data, or both?

6. Look at the condition numbers. Could numerical instability also contribute to the observed behavior?

#### 6. EXPERIMENT — Does more training data help?

Repeat the small-sample experiment using

$$
m_{\mathrm{train}}\in\{60,120,300,1000\}.
$$

For each training size:

1. Increase the polynomial degree $r$.
2. Record the number of parameters $D$.
3. Record training and test $R^2$.
4. Identify where test performance starts to deteriorate.
5. Compare $D$ with the number of training examples.

**Before running, predict:**

If we increase the number of training examples while keeping the same polynomial degree, would you generally expect the risk of overfitting to increase or decrease?

**After running, discuss:**

Does the deterioration occur at the same polynomial degree for every training size?

Can the number of parameters alone explain the results, or should we also consider the condition number and the structure of the data?


---
# Investigation 6 — Implementing SGD for linear regression

Up to here we solved ERM **exactly**, by linear algebra. That is possible for the squared loss, but it is costsly and requires the whole dataset in memory at once. For large $m$ and $d$, and for the losses we will meet later, we use an **iterative** method instead.

### From the slides

The objective is the empirical risk on the training set $S = \{(\mathbf{x}_i, y_i)\}_{i=1}^m$:
$$ L_S(\mathbf{w}) \;=\; \frac{1}{m}\sum_{i=1}^{m} \ell\big(\mathbf{w},(\mathbf{x}_i,y_i)\big), \qquad \ell\big(\mathbf{w},(\mathbf{x},y)\big) = \big(\langle \mathbf{w},\mathbf{x}\rangle - y\big)^2 . $$

**Gradient Descent** takes a step against the gradient of the *whole* objective,
$$ \mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \eta \,\nabla L_S(\mathbf{w}^{(t)}), \qquad \nabla L_S(\mathbf{w}) = \frac{2}{m}\,\mathbf{X}^T(\mathbf{X}\mathbf{w}-\mathbf{y}), $$
which costs $O(md)$ **per step**.

**Stochastic Gradient Descent** picks one training example $i$ at random and takes a step against the gradient of *that example's* loss only:
$$ \boxed{\;\mathbf{w}^{(t+1)} \;=\; \mathbf{w}^{(t)} - \eta \, \nabla_{\mathbf{w}}\,\ell\big(\mathbf{w}^{(t)},(\mathbf{x}_i,y_i)\big) \;=\; \mathbf{w}^{(t)} - \eta \cdot 2\big(\langle \mathbf{w}^{(t)},\mathbf{x}_i\rangle - y_i\big)\,\mathbf{x}_i \;} $$
which costs $O(d)$ per step. The direction is **not** a descent direction in general; it is one *in expectation*, because
$$ \mathbb{E}_{i \sim \text{Unif}[m]}\big[\nabla_{\mathbf{w}}\ell(\mathbf{w},(\mathbf{x}_i,y_i))\big] \;=\; \nabla L_S(\mathbf{w}). $$

Note the shape of the update: it is a multiple of $\mathbf{x}_i$, and the multiplier is proportional to the **residual** $\langle \mathbf{w},\mathbf{x}_i\rangle - y_i$ on that single example. Compare this with the Perceptron update of Lab 2, $\mathbf{w} \leftarrow \mathbf{w} + y_i \mathbf{x}_i$ on a misclassified point: same family, different loss.

### Deriving the gradient — check it yourself

With $f(\mathbf{w}) = (\langle \mathbf{w},\mathbf{x}\rangle - y)^2$, write $u(\mathbf{w}) = \langle \mathbf{w},\mathbf{x}\rangle - y$, so $f = u^2$ and $\nabla u = \mathbf{x}$. By the chain rule
$$ \nabla f(\mathbf{w}) = 2u(\mathbf{w})\,\nabla u(\mathbf{w}) = 2\big(\langle \mathbf{w},\mathbf{x}\rangle - y\big)\,\mathbf{x}. $$

Make sure you can redo this in one line before you code it.

## TO DO: IMPLEMENT SGD FOR LINEAR REGRESSION

The goal is not simply to write working Python code, but to understand **what one stochastic step actually does to the model**.

Everything around the update is already written for you: input handling, the design matrix, the random number generator, the initialization, the loop over epochs, the recording of the loss, and the return statement. **You only have to write the update itself**, inside the banner.

### Step 0 — One convention

As in Part 1, we handle the intercept by prepending a column of ones to the data. So inside the function we work with the design matrix `A`, of shape `(n, d+1)`, and `w` has `d+1` components, with `w[0]` playing the role of the intercept. The $i$-th extended instance is simply `A[i]`.

### Step 1 — Understand what is already there

The function initializes:

* `A`, the design matrix built from `X` by adding the column of ones;
* `w`, a vector of zeros with `A.shape[1]` components;
* `rng`, a local random number generator (exactly as in `my_perceptron` in Lab 2);
* `loss_history`, a list where the training MSE after each epoch is stored.

**Before continuing, check:** how many components does `w` have for this dataset?

### Step 2 — One epoch

An **epoch** is one pass over all the training examples. We pass over them in a **random order**, drawn afresh at every epoch with `rng.permutation(n)`. (Strictly speaking, the version analysed in class draws $i$ uniformly at random *with* replacement; passing over a random permutation is the variant used in practice, and in `scikit-learn`. You will compare the two in one of the experiments.)

### Step 3 — The update *(this is your part)*

For the current example `i`, with extended instance `A[i]` and target `y[i]`:

1. compute the prediction $\langle \mathbf{w}, \mathbf{x}_i\rangle$;
2. compute the residual, prediction minus target — mind the **sign**;
3. compute the gradient $2\,(\text{residual})\,\mathbf{x}_i$ of the loss of this single example;
4. move $\mathbf{w}$ **against** the gradient, by a step of length `eta`.

One to three lines. Do not change anything outside the banner.

### Step 4 — Returning

The function returns `w` and `loss_history`. **Do not change the function name, its parameters, their default values, or the return statement.**

In [ ]:
#do not change the function name, input parameters, and default values
#your implementation should output the weight vector w (np.array of length d+1, with w[0] the
#intercept) and loss_history, the list of the training MSE after each epoch

def my_sgd_linear_regression(X, y, eta=0.001, n_epochs=20, seed=None):
    # -- accept any array-like input and work with genuine float ndarrays from here on
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # -- X must be 2-D (one row per sample, one column per feature) and y 1-D with
    #    exactly one target per row of X
    if X.ndim != 2 or y.shape != (len(X),):
        raise ValueError("Incompatible X and y shapes.")

    # -- the design matrix: a leading column of ones, so that w[0] is the intercept
    n = X.shape[0]
    A = np.hstack([np.ones((n, 1)), X])

    # -- a LOCAL random number generator: an integer seed makes the run reproducible,
    #    seed=None draws fresh entropy. Being local, it cannot be perturbed by other
    #    cells of the notebook drawing random numbers.
    rng = np.random.default_rng(seed)

    # -- initialize the weight vector (intercept included) to all zeros
    w = np.zeros(A.shape[1])

    # -- here we store the training MSE at the end of each epoch
    loss_history = []

    for epoch in range(n_epochs):
        # -- one epoch = one pass over all the training examples, in a random order
        for i in rng.permutation(n):

            # ===================================================================
            # =====================  YOUR CODE STARTS HERE  =====================
            # ===================================================================
            #
            # Perform ONE stochastic gradient step using example i only:
            #
            #     prediction  :  < w , A[i] >
            #     residual    :  < w , A[i] >  -  y[i]
            #     gradient    :  2 * residual * A[i]
            #     update      :  w  <-  w  -  eta * gradient
            #
            # Write 1 to 3 lines below. Do not change anything outside this block.

            raise NotImplementedError("Write the SGD update here, then delete this line.")

            # ===================================================================
            # ======================  YOUR CODE ENDS HERE  ======================
            # ===================================================================

        # -- end of the epoch: record the training MSE of the current w
        loss_history.append(float(np.mean((A @ w - y) ** 2)))

    #do not change the output instruction
    return w, loss_history

### A quick self-check before using it on the real data

Let's build a tiny problem whose answer we know exactly: two features, targets generated *exactly* by a linear function with no noise. If your update is right, SGD must get very close to the true coefficients.

In [ ]:
rng_check = np.random.default_rng(0)
X_check = rng_check.normal(size=(500, 2))
w_true = np.array([3.0, -2.0, 0.5])            # intercept, then the two coefficients
y_check = w_true[0] + X_check @ w_true[1:]      # no noise at all

w_check, hist_check = my_sgd_linear_regression(X_check, y_check, eta=0.01, n_epochs=30, seed=0)

print('true  w:', w_true)
print('SGD   w:', np.round(w_check, 6))
print('\nfinal training MSE:', f'{hist_check[-1]:.3e}', '  (should be essentially 0)')
assert np.allclose(w_check, w_true, atol=1e-3), 'Something is wrong with the update.'
print('\nSelf-check passed.')

## Running SGD on the power plant data

### PREDICT — do not run the next cells yet

1. The exact ERM solution `w_scaled` is already in memory. After $20$ epochs, do you expect SGD to return *exactly* that vector? Approximately? Not at all?
2. Do you expect the training MSE recorded at the end of each epoch to decrease **monotonically**?
3. Which training MSE will be smaller: the one reached by SGD, or the one of the closed-form solution? Can SGD do better?

In [ ]:
eta: float = 0.001
n_epochs: int = 20

w_sgd, loss_history = my_sgd_linear_regression(X_train_scaled, y_train,
                                               eta=eta, n_epochs=n_epochs, seed=42)

print('SGD         w:', np.round(w_sgd, 4))
print('closed form w:', np.round(w_scaled, 4))
print('\n||w_sgd - w_closed_form|| =', f'{norm(w_sgd - w_scaled):.4f}')
print('(for comparison, ||w_closed_form|| =', f'{norm(w_scaled):.4f})')

pred_train_sgd = D_train_scaled @ w_sgd
pred_test_sgd = D_test_scaled @ w_sgd

print('\n%-14s %12s %12s %12s %12s' % ('', 'train MSE', 'train R^2', 'test RMSE', 'test R^2'))
print('%-14s %12.4f %12.4f %12.4f %12.4f'
      % ('SGD', np.mean((pred_train_sgd - y_train) ** 2), r_squared(y_train, pred_train_sgd),
         rmse(y_test, pred_test_sgd), r_squared(y_test, pred_test_sgd)))
print('%-14s %12.4f %12.4f %12.4f %12.4f'
      % ('closed form', np.mean((pred_train - y_train) ** 2), r_squared(y_train, pred_train),
         rmse(y_test, pred_test), r_squared(y_test, pred_test)))

In [ ]:
mse_star = float(np.mean((pred_train - y_train) ** 2))   # the minimum, from the closed form

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, n_epochs + 1), loss_history, 'o-', label='SGD')
plt.axhline(mse_star, color='r', linestyle='--', label='closed-form minimum')
plt.xlabel('epoch'); plt.ylabel('training MSE'); plt.legend(); plt.title('Training MSE per epoch')

plt.subplot(1, 2, 2)
plt.semilogy(range(1, n_epochs + 1), np.array(loss_history) - mse_star, 'o-')
plt.xlabel('epoch'); plt.ylabel('training MSE  -  minimum'); plt.title('Suboptimality (log scale)')
plt.tight_layout(); plt.show()

print('training MSE per epoch:')
print(np.round(loss_history, 4))

### RUN & OBSERVE → EXPLAIN

SGD gets very close to the exact ERM solution, but it does **not** converge to it. With a **constant** step size the iterates do not settle at the minimum: once they are near it, each stochastic gradient is dominated by the noise of the single sampled example, and $\mathbf{w}$ keeps bouncing inside a ball whose radius is proportional to $\eta$. This is why the loss curve flattens out at a level slightly *above* the red line and then wobbles instead of decreasing.

Note also that the closed-form solution **must** have the smaller **training** MSE: it is the exact minimizer of that quantity. SGD cannot beat it — and if your implementation seems to, look for a bug.

But look at the **test** $R^2$: it is essentially the same, and it may well come out marginally *better* for SGD. That is not a contradiction and it is not a bug: nothing says that the exact minimizer of the empirical risk is the better predictor on unseen data. The gap between SGD and the exact minimizer is far smaller than the gap between training error and generalization error. Optimizing the empirical risk beyond a certain accuracy does not necessarily lead to better generalization.

## Experiment A — the step size $\eta$

### PREDICT

The step size is the only real knob of the method. Sketch, on the Thinking Sheet, the loss curve you expect for a step size that is (a) far too small, (b) well chosen, (c) far too large. Where does each curve end up after $20$ epochs?

In [ ]:
etas = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 5e-1]

plt.figure(figsize=(7, 4.5))
print('%-10s %14s %16s %12s' % ('eta', 'final train MSE', '||w - w_closed||', 'test R^2'))
for e in etas:
    with np.errstate(over='ignore', invalid='ignore'):
        w_e, hist_e = my_sgd_linear_regression(X_train_scaled, y_train,
                                               eta=e, n_epochs=20, seed=42)
    if not np.all(np.isfinite(w_e)):
        print('%-10g %14s %16s %12s' % (e, 'overflow', '--', 'DIVERGED'))
        continue
    print('%-10g %14.4f %16.4f %12.4f'
          % (e, hist_e[-1], norm(w_e - w_scaled), r_squared(y_test, D_test_scaled @ w_e)))
    plt.plot(range(1, 21), hist_e, 'o-', markersize=3, label=f'eta = {e:g}')

plt.axhline(mse_star, color='r', linestyle='--', label='closed-form minimum')
plt.yscale('log'); plt.xlabel('epoch'); plt.ylabel('training MSE (log scale)')
plt.legend(fontsize=8); plt.title('Effect of the step size'); plt.tight_layout(); plt.show()

### EXPLAIN

Three regimes, all visible in the table:

* **$\eta$ too small** — every step barely moves $\mathbf{w}$; after $20$ epochs the model is still far from the minimum. Nothing is wrong, there is simply not enough budget.
* **$\eta$ well chosen** — fast progress, then a plateau slightly above the minimum, with a residual wobble proportional to $\eta$.
* **$\eta$ too large** — the iterates overshoot the minimum by more than they started from it, and the magnitude of $\mathbf{w}$ grows at every step until it overflows.

There is a simple way to see *where* the boundary is. Consider one step on example $i$, and look at how the residual on that same example changes:
$$ \langle \mathbf{w}^{(t+1)}, \mathbf{x}_i\rangle - y_i \;=\; \big(1 - 2\eta\lVert \mathbf{x}_i\rVert^2\big)\big(\langle \mathbf{w}^{(t)}, \mathbf{x}_i\rangle - y_i\big). $$
So the step shrinks that residual only if $\lvert 1 - 2\eta\lVert \mathbf{x}_i\rVert^2 \rvert < 1$, that is
$$ \eta \;<\; \frac{1}{\lVert \mathbf{x}_i\rVert^{2}} \qquad\text{for every training example } i . $$
Derive this identity yourself — it is two lines — and then run the next cell.

This is a worst-case, one-example-at-a-time argument, so it is a **heuristic**, not a theorem about the whole run: you will see the method survive step sizes a few times larger than the bound. What it gets right is the *order of magnitude* — and, above all, how that order of magnitude moves when the features are rescaled, which is the subject of the next experiment.

In [ ]:
sq_norms_scaled = (D_train_scaled ** 2).sum(axis=1)
sq_norms_raw = (D_train_raw ** 2).sum(axis=1)

print('SCALED features:  max ||x_i||^2 = %10.4g   ->  heuristic bound eta < %.3g'
      % (sq_norms_scaled.max(), 1.0 / sq_norms_scaled.max()))
print('RAW    features:  max ||x_i||^2 = %10.4g   ->  heuristic bound eta < %.3g'
      % (sq_norms_raw.max(), 1.0 / sq_norms_raw.max()))

## Experiment B — does scaling matter *for SGD*?

### PREDICT — commit before running

In Investigation 3 we saw that scaling does **not** change the closed-form solution: the predictions were identical to $10^{-13}$.

Now we run **exactly the same SGD**, with **exactly the same step size** $\eta = 10^{-3}$, on the **unscaled** features. What do you expect: the same model? a slightly worse one? something else entirely?

In [ ]:
with np.errstate(over='ignore', invalid='ignore'):
    w_unscaled, hist_unscaled = my_sgd_linear_regression(X_train, y_train,
                                                         eta=0.001, n_epochs=20, seed=42)

if np.all(np.isfinite(w_unscaled)):
    print('w on unscaled features:', np.round(w_unscaled, 4))
    print('final training MSE:', hist_unscaled[-1])
else:
    print('SGD on the UNSCALED features with eta = 0.001 DIVERGED.')
    print('w =', w_unscaled)
    print('training MSE after each epoch:', hist_unscaled[:5], '...')

In [ ]:
# -- which step sizes DO work on the unscaled features?
print('%-10s %18s' % ('eta', 'final train MSE'))
for e in [1e-3, 1e-5, 1e-6, 5e-7, 1e-7, 1e-8]:
    with np.errstate(over='ignore', invalid='ignore'):
        w_e, hist_e = my_sgd_linear_regression(X_train, y_train, eta=e, n_epochs=5, seed=42)
    print('%-10g %18s' % (e, f'{hist_e[-1]:.4f}' if np.all(np.isfinite(w_e)) else 'DIVERGED'))

print('\nFor comparison, on the SCALED features with eta = 1e-3 after 5 epochs:')
w_e, hist_e = my_sgd_linear_regression(X_train_scaled, y_train, eta=1e-3, n_epochs=5, seed=42)
print('final train MSE = %.4f   (closed-form minimum = %.4f)' % (hist_e[-1], mse_star))

### EXPLAIN

Put the two halves together:

| | closed form | SGD |
|---|---|---|
| effect of scaling on the **learned function** | none at all (identical to $10^{-13}$) | enormous — it is the difference between working and overflowing |

The **learning problem** is exactly the same in both coordinate systems: same hypothesis class, same ERM minimizer. What changes is the **geometry seen by the algorithm**. The largest usable step size is dictated by the largest feature, while the progress made along the smallest feature is dictated by the smallest one — and the gap between the two is measured by the condition number, which scaling took from $\approx 2\cdot 10^5$ down to $\approx 5$.


 ### CONNECTING WITH LAB 2: the Perceptron

In Lab 2, we observed that different runs of the Perceptron may produce different separating hyperplanes.

There are three important aspects to distinguish.

**A. The separating hyperplane is not necessarily unique**

Recall the Perceptron update rule:

$$
\mathbf{w}^{(t+1)}
=
\mathbf{w}^{(t)}+y_i\mathbf{x}_i
$$

whenever

$$
y_i\langle\mathbf{w}^{(t)},\mathbf{x}_i\rangle\leq 0.
$$

If the training data are linearly separable, the classical Perceptron convergence theorem guarantees that the algorithm eventually finds a separating hyperplane.

However, the theorem does not specify which separating hyperplane will be found.

Any parameter vector satisfying

$$
y_i\langle\mathbf{w},\mathbf{x}_i\rangle>0
\qquad \text{for every training example }i
$$

is a valid separating solution.

There may be infinitely many such solutions.

**B. The order of the examples can affect the result**

The Perceptron updates its weights sequentially.

Each update depends on the current weights, which depend on all previous updates.

Therefore, changing the order in which examples are presented may change the sequence of updates and the final separating hyperplane.

If examples are selected or shuffled randomly, different random seeds may lead to different results.

Fixing the random seed makes the random sequence reproducible, but it does not make the separating hyperplane mathematically unique.

Notice that randomness is not required by the classical Perceptron algorithm: a fixed deterministic order is also possible.

**C. Feature scaling changes the updates**

Consider a simple two-dimensional example:

$$
\mathbf{x}=
\begin{pmatrix}
1\\
2
\end{pmatrix},
\qquad
y=+1,
\qquad
\mathbf{w}^{(0)}=
\begin{pmatrix}
0\\
0
\end{pmatrix}.
$$

The initial signed margin is zero, so the Perceptron performs an update:

$$
\mathbf{w}^{(1)}
=
\mathbf{w}^{(0)}+y\mathbf{x}
=
\begin{pmatrix}
1\\
2
\end{pmatrix}.
$$

Now suppose we rescale the second feature by a factor of 10:

$$
\mathbf{z}=
\begin{pmatrix}
1\\
20
\end{pmatrix}.
$$

Starting again from zero, the update becomes

$$
\widetilde{\mathbf{w}}^{(1)}
=
\begin{pmatrix}
1\\
20
\end{pmatrix}.
$$

But these weight vectors are expressed in different coordinate systems.

To compare the resulting predictors, we must express them in the same coordinates.

In the original coordinates, the first predictor has score

$$
f_1(\mathbf{x})=x_1+2x_2.
$$

The second predictor, trained on scaled features, has score

$$
f_2(\mathbf{x})=x_1+20(10x_2)
=x_1+200x_2.
$$

These scores define different decision boundaries:

$$
x_1+2x_2=0
$$

and

$$
x_1+200x_2=0.
$$

Therefore, even with the same initial weights, the same training example, and no randomness, feature scaling can change the Perceptron update and the resulting decision boundary.

This does not mean that the set of representable linear classifiers has changed.

Since the scaling transformation is invertible, every linear classifier expressed in the scaled coordinates can also be expressed in the original coordinates, and vice versa.

**What changes is the trajectory followed by the Perceptron update rule, not the hypothesis class itself.**

**What about linear regression solved using SGD?**




## Experiment C — choose **one** *(optional)*

Pick **one** of the following, predict the outcome, then test it. The goal is not to find the "best" configuration, but to be able to explain the effect.

1. **Sampling scheme.** Replace `rng.permutation(n)` by drawing each index independently with `rng.integers(n)` ($n$ draws per epoch, *with* replacement — the version analysed in class). Does anything visible change?
2. **Decaying step size.** Replace the constant `eta` by $\eta_t = \eta_0/(1 + t/t_0)$, where $t$ counts the updates. Does the residual wobble around the minimum disappear? At what price?
3. **Averaging.** Keep the constant step size but return the average of the iterates over the second half of the run instead of the last one. Compare $\lVert \mathbf{w} - \mathbf{w}^\star\rVert$.
4. **Initialization.** Start from `w = rng.normal(size=d+1)` instead of zeros. Does the final result change? Does the first part of the loss curve change?
5. **Full-batch Gradient Descent.** Implement one GD step, $\mathbf{w} \leftarrow \mathbf{w} - \eta\,\frac{2}{n}\mathbf{A}^T(\mathbf{A}\mathbf{w}-\mathbf{y})$, and compare *per epoch* (one GD step uses as much computation as one SGD epoch). Which makes more progress for the same cost?

Copy the function into a new cell and modify the copy, so that `my_sgd_linear_regression` stays as it is.

## For reference — the scikit-learn version

`SGDRegressor` is the library implementation of what you just wrote. With `penalty=None` (no regularization — we will discuss it later in the course), `loss='squared_error'` and a constant learning rate, it runs the same algorithm.

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd_sklearn = SGDRegressor(loss='squared_error', penalty=None,
                           learning_rate='constant', eta0=0.001,
                           max_iter=20, tol=None, random_state=42)
sgd_sklearn.fit(X_train_scaled, y_train)

print('scikit-learn SGD  intercept:', np.round(sgd_sklearn.intercept_, 4),
      ' coef:', np.round(sgd_sklearn.coef_, 4))
print('our SGD           w        :', np.round(w_sgd, 4))
print('closed form       w        :', np.round(w_scaled, 4))

print('\ntest R^2, scikit-learn SGD:', round(r2_score(y_test, sgd_sklearn.predict(X_test_scaled)), 4))
print('test R^2, our SGD         :', round(r_squared(y_test, pred_test_sgd), 4))
print('test R^2, closed form     :', round(r_squared(y_test, pred_test), 4))

# End-of-lab retrieval — close the notebook

Without looking back, discuss with your partner:

1. Why does feature scaling leave the closed-form least squares model unchanged, but decide whether SGD converges or explodes?
2. Write down the SGD update for linear regression with squared loss from memory. Where does the factor $2$ come from, and why is the update a multiple of $\mathbf{x}_i$?
3. What does $R^2 = 0$ mean? Why can $R^2$ be negative on the test set?
4. With a constant step size, why does SGD stop improving before reaching the exact minimum — and why does that not hurt the test error?
5. Feature expansion makes the model non-linear in $\mathbf{x}$ but leaves the normal equations unchanged. Why?
6. One result from today that you could explain at the exam without running any code: ______